# Multiple Linear Regression — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 4

---
**Learning Objectives:**
- Extend the regression model from one regressor to $k$ regressors and interpret each slope as a **partial effect** (ceteris paribus)
- Diagnose **omitted-variable bias** by comparing the CAPM (one factor) with the Fama-French three-factor model
- Compute $R^2$ and **adjusted $R^2$** by hand and use adjusted $R^2$ for model selection
- Run the **F-test for joint significance** by hand from $R^2_U$ and $R^2_R$ — and verify with `model.f_test()`
- Detect **multicollinearity** with auxiliary regressions and the **variance inflation factor (VIF)**

> Run each cell with **Shift+Enter**. This notebook accompanies the V6 lecture slides.

## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance statsmodels pandas-datareader --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Part 1 — The Running Example: Apple and the Fama-French 3-Factor Model

The multiple linear regression model with $k$ regressors is

$$y_t = \beta_0 + \beta_1 x_{1,t} + \beta_2 x_{2,t} + \cdots + \beta_k x_{k,t} + u_t$$

Our running example explains **Apple's daily excess return** with THREE factors instead of one:

$$r_{AAPL,t} - r_{f,t} = \beta_0 + \beta_1\,(Mkt\text{-}RF)_t + \beta_2\,SMB_t + \beta_3\,HML_t + u_t$$

- **Mkt-RF** — the market excess return (as in CAPM)
- **SMB** — *Small Minus Big*: the size factor
- **HML** — *High Minus Low*: the value factor

The daily factors come from Kenneth French's public data library — one `pandas-datareader` call.

### 1.1 Download Apple + the three factors

In [ ]:
# sort=True is not cosmetic. pandas 3 still sorts a DatetimeIndex concat by
# default but warns that pandas 4 will not; an unsorted index then breaks every
# downstream .loc['2020-01-01':'2020-06-30'] date slice. Say what we mean.
START = '2019-01-01'
END   = '2024-12-31'

# Apple daily returns (in %)
aapl_px  = yf.download('AAPL', start=START, end=END, auto_adjust=True, progress=False)['Close'].squeeze()
aapl_ret = aapl_px.pct_change().dropna() * 100

# Fama-French daily factors (already in %) from Kenneth French's library
ff = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', START, END)[0]
print('Fama-French factor columns:', list(ff.columns))

# Merge on common dates and build Apple's EXCESS return
data = pd.concat([aapl_ret.rename('AAPL'), ff], axis=1, sort=True).dropna()
data['AAPL_excess'] = data['AAPL'] - data['RF']

n = len(data)
print(f'\nMerged sample: {n} trading days, {data.index[0].date()} → {data.index[-1].date()}')
data[['AAPL_excess', 'Mkt-RF', 'SMB', 'HML']].head().round(3)

### 1.2 Key terminology

| Symbol | Meaning | Here |
|--------|---------|------|
| $k$ | number of regressors (excl. intercept) | 3 |
| $k+1$ | number of estimated parameters | 4 |
| $n-k-1$ | residual degrees of freedom | $n-4$ |

Remember $n-2$ from simple regression? That was exactly $n-k-1$ with $k=1$. **Every estimated parameter costs one degree of freedom.**

---
# Part 2 — Estimating the Model: Same OLS, More Slopes

In matrix form the model is $\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \mathbf{u}$ with the OLS estimator
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y}.$$
You never compute this by hand — `statsmodels` does. It minimises the same sum of squared residuals as in simple regression.

### 2.1 Fit the FF3 regression

In [ ]:
y = data['AAPL_excess']
X = sm.add_constant(data[['Mkt-RF', 'SMB', 'HML']])

model_ff3 = sm.OLS(y, X).fit()
print(model_ff3.summary())

In [ ]:
# Extract the quantities we will use throughout
b_mkt  = model_ff3.params['Mkt-RF']
b_smb  = model_ff3.params['SMB']
b_hml  = model_ff3.params['HML']
k      = 3
df_resid = int(model_ff3.df_resid)          # n − k − 1

print(f'β_Mkt_hat = {b_mkt:.4f}   (t = {model_ff3.tvalues["Mkt-RF"]:.1f})')
print(f'β_SMB_hat = {b_smb:.4f}   (t = {model_ff3.tvalues["SMB"]:.1f})')
print(f'β_HML_hat = {b_hml:.4f}   (t = {model_ff3.tvalues["HML"]:.1f})')
print(f'\nR²      = {model_ff3.rsquared:.4f}')
print(f'adj. R² = {model_ff3.rsquared_adj:.4f}')
print(f'n = {int(model_ff3.nobs)},  k = {k},  n − k − 1 = {df_resid}')

---
# Part 3 — Omitted-Variable Bias: CAPM vs. FF3

**The problem:** if we omit a variable that (i) has its own effect on $y$ and (ii) is correlated with a regressor we do include, the OLS slope on that included regressor is biased — it absorbs part of the omitted variable's effect.

**The bias formula** (for one included and one omitted regressor):

$$\hat{\beta}_1^{\,SLR} \;=\; \beta_1^{\,MLR} \;+\; \beta_2 \cdot \frac{\mathrm{Cov}(x_1, x_2)}{\mathrm{Var}(x_1)}$$

The bias is non-zero exactly when $\beta_2 \neq 0$ **and** $\mathrm{Cov}(x_1,x_2) \neq 0$.

### 3.1 See it live: the market slope drops when SMB and HML enter

In [ ]:
# Restricted model — CAPM only (drop SMB and HML)
X_capm = sm.add_constant(data['Mkt-RF'])
model_capm = sm.OLS(y, X_capm).fit()

b_slr = model_capm.params['Mkt-RF']

print(f'CAPM only  (SLR):  β_Mkt_hat = {b_slr:.4f},  R² = {model_capm.rsquared:.4f}')
print(f'FF3        (MLR):  β_Mkt_hat = {b_mkt:.4f},  R² = {model_ff3.rsquared:.4f}')
print(f'\nDifference: {b_slr:.4f} − {b_mkt:.4f} = {b_slr - b_mkt:+.4f}')
print(f'→ The market slope changes by {abs(b_slr - b_mkt)/abs(b_mkt)*100:.1f}% when SMB and HML are added.')
print('→ Part of what looked like market exposure was size/value exposure in disguise.')

In [ ]:
# Both omitted-variable conditions hold: the factors are correlated with the market
# AND carry their own effect (significant t-stats above).
print('Correlation matrix of the three factors:')
print(data[['Mkt-RF', 'SMB', 'HML']].corr().round(3))
print('\n→ SMB and HML are correlated with Mkt-RF (≠ 0) and have β ≠ 0 → OVB conditions met.')

---
# Part 4 — Interpreting the Coefficients: Partial Effects

The key phrase is **ceteris paribus** — all else equal. Each estimated slope is a partial derivative:

$$\frac{\partial\, E[y \mid x_1, \dots, x_k]}{\partial x_j} = \beta_j$$

**Example A — the market loading.** If the market excess return rises by 1 percentage point *and SMB and HML stay fixed*, Apple's excess return rises by $\hat{\beta}_{Mkt}$ percentage points on average.

**Example B — the value loading.** If HML rises by 1pp (a good day for value stocks) *while the other factors stay fixed*, Apple's excess return **falls** by $|\hat{\beta}_{HML}|$ pp — Apple loads negatively on value: statistically, a growth stock.

**Inference is identical to simple regression:** every t-statistic, p-value, and confidence interval works exactly as in V5 — each coefficient simply uses its own standard error from the multiple regression.

In [ ]:
print('Partial-effect interpretation (1pp move in one factor, others fixed):')
for f in ['Mkt-RF', 'SMB', 'HML']:
    b  = model_ff3.params[f]
    t  = model_ff3.tvalues[f]
    p  = model_ff3.pvalues[f]
    lo, hi = model_ff3.conf_int().loc[f]
    direction = 'rises' if b > 0 else 'falls'
    print(f'\n{f}:  β_hat = {b:+.4f}')
    print(f'   → Apple {direction} by {abs(b):.2f}pp per 1pp factor move (ceteris paribus)')
    print(f'   t = {t:.2f},  p = {p:.2e},  95% CI = [{lo:.3f}, {hi:.3f}]')

---
# Part 5 — $R^2$ vs. Adjusted $R^2$: Choosing Between Models

**The problem:** $R^2$ *mechanically increases* every time you add a regressor — even pure noise. OLS can always exploit a tiny chance correlation to shave a little off RSS.

**The fix:** adjusted $R^2$ penalises the lost degrees of freedom:

$$\bar{R}^2 = 1 - (1 - R^2)\,\frac{n-1}{n-k-1}$$

**Decision rule:** add a regressor only if it **increases** $\bar{R}^2$.

### 5.1 Worked example by hand — first $R^2$, then the adjustment

In [ ]:
# Step 0: the sums of squares from the FF3 regression
y_bar = y.mean()
TSS = ((y - y_bar) ** 2).sum()
RSS = (model_ff3.resid ** 2).sum()

print(f'n = {n},  k = {k}')
print(f'TSS = Σ(y − y_bar)² = {TSS:.4f}')
print(f'RSS = Σ u_hat²      = {RSS:.4f}')

# Step 1: R² from the sums of squares
R2 = 1 - RSS / TSS
print(f'\nStep 1:  R² = 1 − RSS/TSS = 1 − {RSS:.4f}/{TSS:.4f} = {R2:.4f}')
print(f'         → the three factors explain {R2*100:.1f}% of Apple’s daily variation')

# Step 2: only now adjust — penalise for k regressors
penalty = (n - 1) / (n - k - 1)
R2_adj  = 1 - (1 - R2) * penalty
print(f'\nStep 2:  penalty = (n−1)/(n−k−1) = {n-1}/{n-k-1} = {penalty:.4f}')
print(f'         adj. R² = 1 − (1 − {R2:.4f}) · {penalty:.4f} = {R2_adj:.4f}')

# Verify against statsmodels
print(f'\nstatsmodels:  R² = {model_ff3.rsquared:.4f},  adj. R² = {model_ff3.rsquared_adj:.4f}')
print(f'Match? {np.allclose([R2, R2_adj], [model_ff3.rsquared, model_ff3.rsquared_adj])}   ← ✓')
print('\nNote how tiny the penalty is here — abundant df (large n, small k).')
print('With n = 30 and k = 10 the ratio would be 29/19 ≈ 1.53 — a penalty that bites.')

### 5.2 The model ladder — watch $\bar{R}^2$ expose an irrelevant variable

In [ ]:
# Four models: CAPM → +SMB → +HML → +pure noise
np.random.seed(42)
data['NOISE'] = np.random.normal(0, 1, len(data))   # an irrelevant regressor

specs = {
    'SLR (CAPM only)':   ['Mkt-RF'],
    'MLR FF2 (Mkt+SMB)': ['Mkt-RF', 'SMB'],
    'MLR FF3':           ['Mkt-RF', 'SMB', 'HML'],
    'FF3 + irrelevant':  ['Mkt-RF', 'SMB', 'HML', 'NOISE'],
}

rows = []
for name, cols in specs.items():
    m = sm.OLS(y, sm.add_constant(data[cols])).fit()
    rows.append({'Model': name, 'k': len(cols),
                 'R²': m.rsquared, 'adj. R²': m.rsquared_adj})
ladder = pd.DataFrame(rows).set_index('Model')
print(ladder.round(4))

fig, ax = plt.subplots(figsize=(10, 4.5))
xpos = np.arange(len(ladder)); w = 0.38
ax.bar(xpos - w/2, ladder['R²'],      w, color=ORANGE, label='$R^2$')
ax.bar(xpos + w/2, ladder['adj. R²'], w, color=YELLOW, edgecolor=GREY, lw=0.5, label='$\\bar{R}^2$')
ax.set_xticks(xpos); ax.set_xticklabels(ladder.index, fontsize=9)
ax.set_ylim(ladder['adj. R²'].min() - 0.02, ladder['R²'].max() + 0.01)
ax.legend(loc='upper left', frameon=False)
ax.set_ylabel('Goodness of fit')
ax.set_title('$R^2$ always grows — $\\bar{R}^2$ grows ONLY if the variable adds real signal',
             fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

print('→ Adding SMB and HML: both bars rise — real signal.')
print('→ Adding NOISE: R² creeps up mechanically, adj. R² is flat or falls — irrelevant variable exposed.')

---
# Part 6 — The F-Test: Are Several Coefficients Jointly Zero?

A **t-test** asks about ONE coefficient — e.g. $H_0: \beta_{SMB} = 0$.

An **F-test** asks about SEVERAL coefficients JOINTLY — e.g.

$$H_0: \beta_{SMB} = \beta_{HML} = 0 \qquad \text{vs.} \qquad H_1: \text{at least one} \neq 0$$

Why not two separate t-tests? (i) the estimates are correlated, and (ii) every extra 5%-test inflates the overall false-rejection rate. The F-test asks one clean question with one clean error rate.

**Mechanics — two regressions:**
- RESTRICTED: impose $H_0$ (drop the $q$ variables) → $R^2_R$
- UNRESTRICTED: full model → $R^2_U$

$$F = \frac{(R_U^2 - R_R^2)\,/\,q}{(1 - R_U^2)\,/\,(n-k-1)} \;\sim\; F_{q,\;n-k-1}$$

**Notation:** $n$ = observations • $k$ = regressors in the unrestricted model (excl. intercept) • $q$ = restrictions under $H_0$ • $n-k-1$ = residual df.

### 6.1 Worked example by hand — do SMB and HML jointly matter for Apple?

In [ ]:
R2_U = model_ff3.rsquared      # unrestricted: FF3
R2_R = model_capm.rsquared     # restricted:   CAPM (H0 imposed)
q    = 2                       # two restrictions: β_SMB = 0 and β_HML = 0

numerator   = (R2_U - R2_R) / q
denominator = (1 - R2_U) / df_resid
F_obs       = numerator / denominator

print(f'R²_U (FF3)  = {R2_U:.4f}')
print(f'R²_R (CAPM) = {R2_R:.4f}')
print(f'\nNumerator:   (R²_U − R²_R)/q       = ({R2_U:.4f} − {R2_R:.4f})/2 = {numerator:.6f}')
print(f'             → lost fit per imposed restriction')
print(f'Denominator: (1 − R²_U)/(n−k−1)    = {1-R2_U:.4f}/{df_resid} = {denominator:.6f}')
print(f'             → unexplained variance per residual degree of freedom')
print(f'\nF_obs = {numerator:.6f} / {denominator:.6f} = {F_obs:.1f}')

# Critical value and decision
F_crit = stats.f.ppf(0.95, q, df_resid)
p_val  = 1 - stats.f.cdf(F_obs, q, df_resid)
print(f'\nF_crit (α = 5%, df = {q}, {df_resid}) = {F_crit:.2f}')
print(f'p-value = {p_val:.2e}')
if F_obs > F_crit:
    print(f'\nF_obs = {F_obs:.1f} > {F_crit:.2f} = F_crit   →   REJECT H0')
    print('→ Apple’s returns ARE driven by size and value beyond the market alone.')
else:
    print(f'\nF_obs = {F_obs:.1f} < {F_crit:.2f} = F_crit   →   do NOT reject H0')
    print('→ On this sample, size and value add nothing beyond the market factor.')

In [ ]:
# Verify with the built-in F-test — same statistic, same p-value
print(model_ff3.f_test('SMB = HML = 0'))
print('\nNote: the F-statistic in model.summary() tests ALL slopes jointly zero')
print(f'(H0: β_Mkt = β_SMB = β_HML = 0):  F = {model_ff3.fvalue:.1f}')

---
# Part 7 — Multicollinearity & the VIF

When regressors carry **overlapping information**, OLS struggles to attribute the effect to any single one.

- **Perfect** multicollinearity: one regressor is an exact linear combination of others → OLS cannot be estimated.
- **Near** multicollinearity: high but imperfect correlation → OLS runs, but SEs inflate.

**Symptoms:** high $R^2$ yet few significant t-stats • coefficients flip when a regressor is added/dropped • wide CIs.

**The VIF diagnostic** — two steps:
1. **Auxiliary regression:** regress $x_j$ on all *other* regressors (not on $y$!) → $R_j^2$
2. $$VIF_j = \frac{1}{1 - R_j^2}, \qquad \sqrt{VIF_j} = \text{SE inflation factor}$$

Rule of thumb: $VIF > 5$ caution, $VIF > 10$ problematic.

### 7.1 VIF by hand for SMB — the auxiliary regression

In [ ]:
# Step 1: auxiliary regression — SMB on the OTHER regressors (y appears nowhere!)
X_aux = sm.add_constant(data[['Mkt-RF', 'HML']])
aux   = sm.OLS(data['SMB'], X_aux).fit()
R2_smb = aux.rsquared
print(f'Step 1:  auxiliary regression SMB ~ Mkt-RF + HML')
print(f'         R²_SMB = {R2_smb:.3f}')
print(f'         → {R2_smb*100:.0f}% of SMB’s variation is already contained in the other two factors')

# Step 2: translate into the VIF
VIF_smb = 1 / (1 - R2_smb)
print(f'\nStep 2:  VIF_SMB = 1/(1 − {R2_smb:.3f}) = {VIF_smb:.2f}')

# Step 3: interpret as SE inflation
print(f'\nStep 3:  √VIF = {np.sqrt(VIF_smb):.2f} → SMB’s standard error is only '
      f'{(np.sqrt(VIF_smb)-1)*100:.0f}% larger than with zero overlap — harmless.')
print('         (If R²_j were 0.90: VIF = 10 → SE ×3.2 — that is where the rule of thumb bites.)')

In [ ]:
# All three factors at once — by hand and with the statsmodels helper
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_no_const = data[['Mkt-RF', 'SMB', 'HML']]
X_with_c   = sm.add_constant(X_no_const)

rows = []
for j, col in enumerate(X_no_const.columns):
    others = [c for c in X_no_const.columns if c != col]
    r2j = sm.OLS(X_no_const[col], sm.add_constant(X_no_const[others])).fit().rsquared
    vif_builtin = variance_inflation_factor(X_with_c.values, j + 1)   # +1 skips the constant
    rows.append({'Regressor': col, 'R²_j': round(r2j, 3),
                 'VIF (by hand)': round(1/(1-r2j), 2),
                 'VIF (statsmodels)': round(vif_builtin, 2)})
print(pd.DataFrame(rows).set_index('Regressor'))
print('\n→ All far below 5 — multicollinearity is no concern in the FF3 model.')

### 7.2 What near-collinearity does to standard errors — a simulation

In [ ]:
# Two worlds with the same true betas — orthogonal vs. highly collinear regressors
np.random.seed(7)
n_sim = 500
x1 = np.random.normal(0, 1, n_sim)

x2_orth = np.random.normal(0, 1, n_sim)                     # corr ≈ 0
x2_coll = 0.95 * x1 + np.random.normal(0, 0.31, n_sim)      # corr ≈ 0.95

results = {}
for label, x2 in [('orthogonal (corr ≈ 0)', x2_orth), ('collinear (corr ≈ 0.95)', x2_coll)]:
    y_sim = 1 + 0.5 * x1 + 0.5 * x2 + np.random.normal(0, 1, n_sim)
    Xs = sm.add_constant(np.column_stack([x1, x2]))
    m  = sm.OLS(y_sim, Xs).fit()
    r2j = sm.OLS(x2, sm.add_constant(x1)).fit().rsquared
    results[label] = {'corr(x1,x2)': np.corrcoef(x1, x2)[0,1],
                      'SE(β1_hat)': m.bse[1], 'SE(β2_hat)': m.bse[2],
                      'VIF': 1/(1-r2j)}
print(pd.DataFrame(results).T.round(3))
print('\n→ Same data-generating process, same n — but the collinear world has')
print('  standard errors roughly √VIF ≈ 3× larger. The overlap destroys precision.')

---
## Summary Table

| Concept | Key Formula | Python (after `model = sm.OLS(y, X).fit()`) |
|---------|-------------|------------------|
| MLR model | $y = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k + u$ | `sm.add_constant(df[cols])` |
| Partial effect | $\partial E[y\mid \cdot]/\partial x_j = \beta_j$ | `model.params` |
| Omitted-variable bias | $\hat{\beta}_1^{SLR} = \beta_1^{MLR} + \beta_2\,\mathrm{Cov}(x_1,x_2)/\mathrm{Var}(x_1)$ | compare nested fits |
| Adjusted $R^2$ | $\bar{R}^2 = 1-(1-R^2)\frac{n-1}{n-k-1}$ | `model.rsquared_adj` |
| F-test (joint) | $F = \frac{(R_U^2-R_R^2)/q}{(1-R_U^2)/(n-k-1)}$ | `model.f_test('SMB = HML = 0')` |
| Auxiliary $R_j^2$ | regress $x_j$ on other regressors | `sm.OLS(xj, X_others).fit().rsquared` |
| VIF | $VIF_j = 1/(1-R_j^2)$ | `variance_inflation_factor(X.values, j)` |

**Diagnostic cheat sheet:**
- $\bar{R}^2$ flat or falling when a variable is added → the variable does not pay for itself
- $F_{obs} > F_{crit}$ → the group of variables jointly matters
- High $R^2$ + weak t-stats → suspect multicollinearity → compute VIFs

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Dummy Variables — bringing qualitative information into the regression.*